# Mouna — GPU training on Colab

**Before running:** `Runtime -> Change runtime type -> GPU` (needs Colab Pro for a reliably fast GPU — T4 is free-tier, Pro gets you better queue priority and A100/L4 access).

**One manual step you need to do first:** upload the dataset parquet file to your Google Drive, in a folder called `mouna_data/`. The file this notebook expects by default is `keypoints_official_split_v3motion.parquet` (our latest, best-tested dataset — 401MB) from `~/mouna/mouna/data/silver/keypoints/` on your Mac. Drag it into `My Drive/mouna_data/` via drive.google.com before running the Drive-mount cell below.

In [ ]:
# 1. Clone the repo (public code only — data is never committed to git)
!git clone https://github.com/vimalthomas/mouna.git
%cd mouna

In [ ]:
# 2. Install the minimal deps actually needed for training (not the full MediaPipe extraction stack — Colab doesn't need that)
!pip install -q torch pandas pyarrow scikit-learn mlflow scipy

In [ ]:
# 3. Mount Drive and link the dataset into the path train_transformer.py expects
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('data/silver/keypoints', exist_ok=True)
!cp /content/drive/MyDrive/mouna_data/keypoints_official_split_v3motion.parquet data/silver/keypoints/

In [ ]:
# 4. Confirm GPU is actually visible to torch before spending a run on it
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — check Runtime > Change runtime type')

In [ ]:
# 5. Train — same script, same env-var pattern as local runs, now picking up CUDA automatically
# (train_transformer.py already checks torch.cuda.is_available() first, MPS second, CPU last)
import os
os.environ['PYTHONPATH'] = 'src'
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
os.environ['MOUNA_PARQUET'] = 'data/silver/keypoints/keypoints_official_split_v3motion.parquet'
os.environ['MOUNA_MODEL_OUT'] = 'data/best_model_colab.pth'
os.environ['MOUNA_RUN_NAME'] = 'pose-transformer-colab-gpu'
os.environ['MOUNA_SEED'] = '42'

!PYTHONPATH=src MLFLOW_ALLOW_FILE_STORE=true python3 scripts/train_transformer.py

## Multi-seed sweep (once a single run looks good)

For the real, statistically-validated comparison (same discipline as every result in this project), run the cell below to sweep seeds 42, 1, 2, 3, 4 — same pattern as `scripts/run_seed_sweep.sh` locally, just inline here since Colab's shell cells don't persist a long-running background process cleanly.

In [ ]:
results = {}
for seed in [42, 1, 2, 3, 4]:
    os.environ['MOUNA_SEED'] = str(seed)
    os.environ['MOUNA_MODEL_OUT'] = f'data/sweep_colab_seed{seed}.pth'
    os.environ['MOUNA_RUN_NAME'] = f'pose-transformer-colab-sweep-seed{seed}'
    !PYTHONPATH=src MLFLOW_ALLOW_FILE_STORE=true python3 scripts/train_transformer.py 2>&1 | tee /tmp/sweep_seed{seed}.log
    import re
    log = open(f'/tmp/sweep_seed{seed}.log').read()
    m = re.search(r'Test acc: ([\d.]+)%', log)
    results[seed] = float(m.group(1)) if m else None
    print(f'seed {seed}: {results[seed]}%')

print()
print('All results:', results)

In [ ]:
# 6. Download the best checkpoint back to your Mac when done
from google.colab import files
files.download('data/best_model_colab.pth')